# ADAS multi-task YOLO26: train on Colab

<a href="https://colab.research.google.com/github/pirazor/YoloMulti/blob/main/colab/train_adas_mt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

One network, three outputs, NMS-free: **object detection** (9 traffic classes), **drivable area**
(direct / alternative) and **lane markings** (solid / dashed), at the 384x640 input the Jetson Orin Nano runs.
This notebook takes you from a dataset to a trained, exported model and an early-results report:

1. configure (data source, preset, model size)
2. install, check the GPU
3. prepare the data (Supervisely export, an already converted dataset, or a synthetic demo set)
4. train (resumable after a Colab disconnect when runs are saved to Google Drive)
5. **early results**: metric curves, best epoch, per-class numbers, prediction overlays
6. export ONNX and check that the exported graph scores the same as the PyTorch weights

**Runtime**: *Runtime → Change runtime type → GPU* (T4 works; L4 / A100 are 2-4x faster).
Then *Runtime → Run all*. To try the pipeline without your data, leave `DATA_SOURCE = "synthetic"`
(~10 min on a T4: the curves must rise within a few epochs, which proves the setup trains).

| preset | what it trains | T4 time (yolo26s) |
|---|---|---|
| `quick` | up to 3,000 random train images, 500 val images, 30 epochs | ~1-1.5 h |
| `full` | everything, 100 epochs (`adas_mt/cfg/default.yaml`) | many hours: use Drive + resume, or L4/A100 |

Optional secrets (key icon in the left sidebar): `GITHUB_TOKEN` if the repository is private,
`HF_TOKEN` for the DINOv3 teacher when `DISTILL = True`.

## 1 · Configuration
`DATA_SOURCE`:
* `synthetic`: generated demo frames (no data needed): a smoke test of the whole pipeline.
* `supervisely_archive`: a `.zip` / `.tar` of a Supervisely export (e.g. on Drive). `DATA_PATH` = the archive.
* `supervisely_dir`: an unpacked Supervisely export folder (copied to the local disk first).
* `converted_archive` / `converted_dir`: a dataset already converted by `python -m adas_mt convert`
  (`images/`, `labels_det/`, `labels_da/`, `labels_ll/`, `data.yaml`); the fastest way to restart a session.

Set `GROUP_REGEX` when frame names carry a clip id (e.g. `'^([0-9a-f]{8})-'` for BDD-style names) so
consecutive frames of one clip never land in both train and val.

In [ ]:
#@title Configuration { display-mode: "form" }
DATA_SOURCE = "synthetic"  #@param ["synthetic", "supervisely_archive", "supervisely_dir", "converted_archive", "converted_dir"]
DATA_PATH = ""  #@param {type:"string"}
PRESET = "quick"  #@param ["quick", "full"]
SCALE = "s"  #@param ["n", "s"]
EPOCHS = 0  #@param {type:"integer"}
BATCH = 0  #@param {type:"integer"}
DISTILL = False  #@param {type:"boolean"}
TEACHER = "dinov3_s"  #@param ["dinov3_s", "dinov3_s_plus", "dinov3_b"]
TEACHER_CKPT = ""  #@param {type:"string"}
RUN_NAME = "colab"  #@param {type:"string"}
RESUME = True  #@param {type:"boolean"}
USE_DRIVE = True  #@param {type:"boolean"}
DRIVE_DIR = "/content/drive/MyDrive/adas_mt"  #@param {type:"string"}
SAVE_CONVERTED_TO_DRIVE = False  #@param {type:"boolean"}
GROUP_REGEX = ""  #@param {type:"string"}
PARTIAL_ANNOTATION = True  #@param {type:"boolean"}
REPO_URL = "https://github.com/pirazor/YoloMulti.git"  #@param {type:"string"}
REPO_REF = "main"  #@param {type:"string"}

# EPOCHS / BATCH = 0 -> preset / GPU-memory defaults. RESUME continues an interrupted run of the same RUN_NAME.
# Advanced (rarely changed):
VAL_FRACTION = 0.1          # share of clips/frames held out when the export has no train/val folders
QUICK_TRAIN_IMAGES = 3000   # quick preset: random subset of the train split
QUICK_VAL_IMAGES = 500      # quick preset: random subset of the val split (validated every epoch)
EVAL_IMAGES = 64            # export check: .pt vs ONNX on this many val images
SYNTHETIC_FRAMES = 0        # 0 -> 600 frames (quick) / 2000 (full)
WORK_DIR = "/content"       # local disk: data, clone, weights
REPO_DIR = ""               # use an existing checkout instead of cloning
DEVICE = ""                 # "" -> GPU 0 if present, else CPU
TRAIN_OVERRIDES = {}        # any Ultralytics train argument, e.g. {"lr0": 0.002, "mosaic": 0.5}
MT_OVERRIDES = {}           # multi-task settings, e.g. {"loss_gains": {"da": 3.0, "ll": 5.0}} (see docs/training.md)

## 2 · Install and check the runtime

In [ ]:
import json, os, random, shlex, shutil, subprocess, sys, time
from pathlib import Path

q = shlex.quote  # every path in a shell command is quoted (Drive folders may contain spaces)

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False


def sh(cmd, check=True, show=None):
    """Run a shell command with its output streamed into the cell; raise when it fails."""
    print(f"$ {show or cmd}", flush=True)
    try:
        ip = get_ipython()  # noqa: F821 (IPython / Colab)
    except NameError:
        ip = None
    if ip is not None:
        ip.system(cmd)
        rc = ip.user_ns.get("_exit_code", 0)
    else:
        rc = subprocess.call(cmd, shell=True)
    if check and rc != 0:
        raise RuntimeError(f"command failed with exit code {rc}: {show or cmd}")
    return rc


def secret(name):
    """Environment variable, else a Colab secret (left sidebar, key icon), else None."""
    if os.environ.get(name):
        return os.environ[name]
    if IN_COLAB:
        try:
            from google.colab import userdata
            return userdata.get(name)
        except Exception:
            return None
    return None


WORK = Path(WORK_DIR)
WORK.mkdir(parents=True, exist_ok=True)
os.chdir(WORK)
if IN_COLAB and (USE_DRIVE or str(DATA_PATH).startswith("/content/drive")):
    from google.colab import drive
    drive.mount("/content/drive")

# --- the code
if REPO_DIR:
    REPO = Path(REPO_DIR).resolve()
else:
    REPO = WORK / "YoloMulti"
    url, token = REPO_URL, secret("GITHUB_TOKEN")
    if token and url.startswith("https://github.com/"):
        url = url.replace("https://", f"https://x-access-token:{token}@")
    hidden = url.replace(token, "***") if token else url
    if (REPO / ".git").exists():
        sh(f"git -C {q(str(REPO))} fetch -q --depth 1 {q(url)} {q(REPO_REF)} && git -C {q(str(REPO))} checkout -q FETCH_HEAD",
           show=f"git fetch {hidden} {REPO_REF}")
    else:
        sh(f"git clone -q --depth 1 --branch {q(REPO_REF)} {q(url)} {q(str(REPO))}", show=f"git clone {hidden} {REPO}")
assert (REPO / "adas_mt" / "__init__.py").exists(), f"no adas_mt package in {REPO}: check REPO_URL / GITHUB_TOKEN"
# the notebook may be newer than the code it checked out (e.g. opened from a branch while REPO_REF is still "main")
NEEDS = ["adas_mt/data/synthetic.py", "adas_mt/data/convert_supervisely.py", "adas_mt/export.py", "adas_mt/deploy/evaluate.py"]
missing = [f for f in NEEDS if not (REPO / f).exists()]
if missing:
    raise SystemExit(f"the code checked out at REPO_REF={REPO_REF!r} is older than this notebook (missing {missing}). "
                     "Set REPO_REF to the branch this notebook came from (e.g. 'multitask'), then Runtime -> Restart session "
                     "and Run all.")
sys.path.insert(0, str(REPO))
os.environ["PYTHONPATH"] = str(REPO) + os.pathsep + os.environ.get("PYTHONPATH", "")

# --- dependencies (torch, OpenCV, numpy come with Colab; never reinstall torch)
sh(f'{q(sys.executable)} -m pip install -q "ultralytics==8.4.171" "timm>=1.0.20" onnx onnxruntime onnxslim')

from ultralytics import settings as yolo_settings  # noqa: E402

# experiment trackers that may be installed on the VM would otherwise ask for logins / API keys mid-run
yolo_settings.update({k: False for k in ("wandb", "mlflow", "comet", "clearml", "dvc", "raytune") if k in yolo_settings})
print("code:", REPO, "| commit", subprocess.run(["git", "-C", str(REPO), "rev-parse", "--short", "HEAD"],
                                              capture_output=True, text=True).stdout.strip() or "?")

In [ ]:
import torch

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    GPU_GB = props.total_memory / 2**30
    print(f"GPU: {props.name}, {GPU_GB:.0f} GB, compute {props.major}.{props.minor}")
    DEVICE = DEVICE or "0"
else:
    GPU_GB = 0.0
    print("No GPU. On Colab: Runtime -> Change runtime type -> GPU, then Runtime -> Run all.")
    if IN_COLAB and not (DATA_SOURCE == "synthetic" and PRESET == "quick"):
        raise SystemExit("refusing to train on real data on a CPU runtime (it would take days)")
    DEVICE = DEVICE or "cpu"

ON_GPU = DEVICE != "cpu"
# bf16 on Ampere+ (A100/L4/H100): no fp16 overflow and no AMP check; fp16 AMP on T4/V100
AMP = "bf16" if ON_GPU and torch.cuda.is_bf16_supported(including_emulation=False) else ON_GPU
if not BATCH:
    BATCH = 64 if GPU_GB >= 30 else 32 if GPU_GB >= 14 else 16 if ON_GPU else 4
WORKERS = min(8, os.cpu_count() or 2)
if not EPOCHS:
    EPOCHS = 30 if PRESET == "quick" else 100
print(f"plan: yolo26{SCALE} | {PRESET} | {EPOCHS} epochs | batch {BATCH} | amp {AMP} | {WORKERS} dataloader workers"
      f" | distillation {'on (' + TEACHER + ')' if DISTILL else 'off'} | device {DEVICE}")

## 3 · Data
The dataset always lands in `/content/data/adas` (a fixed path, so a resumed run finds it again). The converter
turns Supervisely rectangles into YOLO boxes and polygons / lines / bitmaps into the drivable-area and lane masks;
with `PARTIAL_ANNOTATION` a frame without lane (or drivable-area) objects is treated as *not annotated* for that task
instead of *all background*.

In [ ]:
import yaml

from adas_mt.data.convert_supervisely import convert

DATA = WORK / "data" / "adas"
RAW = WORK / "data" / "raw"
recipe = {"source": DATA_SOURCE, "path": DATA_PATH, "partial": PARTIAL_ANNOTATION, "group": GROUP_REGEX,
          "val_fraction": VAL_FRACTION}
if DATA_SOURCE == "synthetic":
    recipe["synthetic"] = SYNTHETIC_FRAMES or (600 if PRESET == "quick" else 2000)
marker = DATA / ".prepared.json"


def unpack(archive, dst):
    archive = Path(archive)
    if not archive.is_file():
        raise FileNotFoundError(f"DATA_PATH {archive} does not exist (is Drive mounted / the path right?)")
    local = WORK / "data" / archive.name
    if archive.resolve() != local.resolve():
        print(f"copying {archive} -> {local}")
        shutil.copyfile(archive, local)  # one sequential read from Drive is much faster than unpacking through it
    print(f"unpacking {local.name}")
    shutil.unpack_archive(str(local), str(dst))
    if local != archive:
        local.unlink()


def find_root(base, name):
    """Shallowest directory under base that contains `name`."""
    hits = sorted(base.rglob(name), key=lambda p: len(p.parts))
    if not hits:
        raise FileNotFoundError(f"no {name} under {base}")
    return hits[0].parent


if marker.exists() and json.loads(marker.read_text()) == recipe:
    print(f"dataset already prepared in {DATA}")
else:
    shutil.rmtree(DATA, ignore_errors=True)
    shutil.rmtree(RAW, ignore_errors=True)
    (WORK / "data").mkdir(parents=True, exist_ok=True)
    t0 = time.time()
    if DATA_SOURCE.startswith("supervisely") or DATA_SOURCE == "synthetic":
        group = GROUP_REGEX or None
        if DATA_SOURCE == "synthetic":
            from adas_mt.data.synthetic import GROUP_REGEX as SYN_GROUP, make_supervisely

            print(f"generating {recipe['synthetic']} synthetic 1280x720 frames")
            make_supervisely(RAW, recipe["synthetic"], seed=0)
            group = group or SYN_GROUP
        elif DATA_SOURCE == "supervisely_archive":
            unpack(DATA_PATH, RAW)
        else:
            print(f"copying {DATA_PATH} -> {RAW}")
            shutil.copytree(DATA_PATH, RAW)
        stats = convert(src=RAW, dst=DATA, val_fraction=VAL_FRACTION, partial_annotation=PARTIAL_ANNOTATION,
                        group_regex=group, overwrite=True)
        print(f"converted {stats.images} images, {stats.boxes} boxes, {stats.da_polys} drivable-area and "
              f"{stats.ll_polys} lane objects")
        for what, n in (("unknown classes", dict(stats.skipped_unknown_class)), ("undrawable geometry", dict(stats.skipped_geometry)),
                        ("unmapped attributes", stats.skipped_unmapped), ("degenerate shapes", stats.skipped_degenerate),
                        ("annotation/image size mismatches", stats.size_mismatch)):
            if n:
                print(f"  note: skipped {what}: {n}")
        shutil.rmtree(RAW, ignore_errors=True)
        if SAVE_CONVERTED_TO_DRIVE and USE_DRIVE:
            dst = Path(DRIVE_DIR) / "datasets" / "adas_converted"
            dst.parent.mkdir(parents=True, exist_ok=True)
            print(f"saving the converted dataset to {dst}.tar (next session: DATA_SOURCE='converted_archive')")
            shutil.make_archive(str(dst), "tar", root_dir=DATA)
    else:
        tmp = WORK / "data" / "unpacked"
        shutil.rmtree(tmp, ignore_errors=True)
        if DATA_SOURCE == "converted_archive":
            unpack(DATA_PATH, tmp)
        else:
            print(f"copying {DATA_PATH} -> {tmp}")
            shutil.copytree(DATA_PATH, tmp)
        shutil.move(str(find_root(tmp, "data.yaml")), str(DATA))
        shutil.rmtree(tmp, ignore_errors=True)
    cfg = yaml.safe_load((DATA / "data.yaml").read_text())
    cfg["path"] = str(DATA)  # the dataset may have been converted elsewhere
    (DATA / "data.yaml").write_text(yaml.safe_dump(cfg, sort_keys=False))
    marker.write_text(json.dumps(recipe))
    print(f"data ready in {time.time() - t0:.0f} s")

In [ ]:
import pandas as pd

DATA_CFG = yaml.safe_load((DATA / "data.yaml").read_text())
IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


def split_images(split):
    d = DATA / "images" / split
    return sorted(p for p in d.iterdir() if p.suffix.lower() in IMG_EXT) if d.is_dir() else []


rows, boxes = [], {}
for split in ("train", "val"):
    ims = split_images(split)
    rows.append({"split": split, "images": len(ims),
                 "drivable-area masks": sum((DATA / "labels_da" / split / f"{p.stem}.png").exists() for p in ims),
                 "lane masks": sum((DATA / "labels_ll" / split / f"{p.stem}.png").exists() for p in ims)})
    for p in ims:
        f = DATA / "labels_det" / split / f"{p.stem}.txt"
        for line in (f.read_text().split("\n") if f.exists() else []):
            if line.strip():
                name = DATA_CFG["names"][int(line.split()[0])]
                boxes.setdefault(name, {"train": 0, "val": 0})[split] += 1
print(pd.DataFrame(rows).to_string(index=False))
print()
print(pd.DataFrame(boxes).T.rename_axis("boxes per class").to_string())
if not rows[0]["images"]:
    raise SystemExit("no training images: check DATA_SOURCE / DATA_PATH")
if not rows[1]["images"]:
    print("\nNOTE: no val split: validation uses the train images, so metrics will be optimistic")

In [ ]:
# Run-specific views of the dataset: the quick preset trains / validates on fixed random subsets (same files on
# every resume), and the export check uses EVAL_IMAGES val images.
def subset(images, n, name):
    if len(images) <= n:
        return None
    pick = sorted(random.Random(0).sample(images, n))
    path = DATA / f"{name}.txt"
    path.write_text("\n".join(str(p) for p in pick) + "\n")
    return str(path)


train_ims, val_ims = split_images("train"), split_images("val") or split_images("train")
run_cfg = dict(DATA_CFG, val=DATA_CFG.get("val") or "images/train")
if PRESET == "quick":
    run_cfg["train"] = subset(train_ims, QUICK_TRAIN_IMAGES, "train_quick") or run_cfg["train"]
    run_cfg["val"] = subset(val_ims, QUICK_VAL_IMAGES, "val_quick") or run_cfg["val"]
RUN_DATA = DATA / "data_run.yaml"
RUN_DATA.write_text(yaml.safe_dump(run_cfg, sort_keys=False))
EVAL_DATA = DATA / "data_eval.yaml"
EVAL_DATA.write_text(yaml.safe_dump(dict(DATA_CFG, val=subset(val_ims, EVAL_IMAGES, "val_eval") or run_cfg["val"]), sort_keys=False))
print(f"train on {run_cfg['train']}, validate on {run_cfg['val']}")

### Optional: DINOv3 teacher (`DISTILL = True`)
The teacher only runs during training and never ships. Its weights come from the Hugging Face hub: accept the DINOv3
licence on the model page (e.g. `huggingface.co/timm/vit_small_patch16_dinov3.lvd1689m`) and add an `HF_TOKEN`
secret, or put Meta's `dinov3_vit*16_pretrain_lvd1689m-*.pth` on Drive and set `TEACHER_CKPT`. This cell loads the
teacher once so a missing licence fails now, not after the data loaders started. Distillation adds roughly 40%
(ViT-S) to 100% (ViT-B) to the training time.

In [ ]:
if DISTILL:
    if secret("HF_TOKEN"):
        os.environ["HF_TOKEN"] = secret("HF_TOKEN")
    probe = (f"import torch; from adas_mt.distill import FrozenTeacher; "
             f"t = FrozenTeacher({TEACHER!r}, pretrained={not TEACHER_CKPT}, checkpoint={TEACHER_CKPT or None!r}); "
             f"x, g = t(torch.rand(1, 3, 384, 640)); assert torch.isfinite(x).all(); print('teacher OK:', t.name, 'grid', g)")
    if sh(f"{q(sys.executable)} -c {q(probe)}", check=False):
        raise SystemExit("the DINOv3 teacher could not be loaded: accept the licence on its Hugging Face page and add "
                         "an HF_TOKEN secret, or set TEACHER_CKPT; or set DISTILL = False")
else:
    print("distillation off")

## 4 · Train
Runs `python -m adas_mt train` (the Ultralytics trainer with the multi-task extras). With `USE_DRIVE` the run directory
is on Google Drive, so after a disconnect: reconnect, *Run all* with the same `RUN_NAME`, and the run resumes from its
last epoch. Each epoch prints detection P/R/mAP plus the drivable-area and lane IoUs.

In [ ]:
import torch
from ultralytics.utils.downloads import attempt_download_asset

PROJECT = (Path(DRIVE_DIR) if USE_DRIVE else WORK) / "runs"
PROJECT.mkdir(parents=True, exist_ok=True)


def run_dirs():
    return sorted((d for d in PROJECT.glob(f"{RUN_NAME}*") if d.is_dir() and (d.name == RUN_NAME or d.name[len(RUN_NAME):].isdigit())),
                  key=lambda d: d.stat().st_mtime)


def resumable():
    """last.pt of the newest run with this name that was interrupted (a finished run's last.pt has epoch -1)."""
    for d in reversed(run_dirs()):
        last = d / "weights" / "last.pt"
        if last.exists():
            try:
                if torch.load(last, map_location="cpu", weights_only=False).get("epoch", -1) >= 0:
                    return last
            except Exception as e:
                print(f"{last} is unreadable ({e}): not resuming from it")
            return None
    return None


base = yaml.safe_load((REPO / "adas_mt" / "cfg" / "default.yaml").read_text())
train = dict(base["train"], epochs=EPOCHS, batch=BATCH, workers=WORKERS, amp=AMP, cache="ram", plots=True,
             deterministic=False, exist_ok=False)  # cache: Ultralytics falls back to disk reads when RAM is short
if PRESET == "quick":
    train.update(close_mosaic=min(5, max(EPOCHS // 5, 1)), patience=EPOCHS, warmup_epochs=1)
train.update(TRAIN_OVERRIDES)
mt = dict(base["mt"], scale=SCALE, **{k: v for k, v in MT_OVERRIDES.items() if k != "distill"})
mt["distill"] = dict(mt.get("distill") or {}, enabled=bool(DISTILL), teacher=TEACHER, teacher_ckpt=TEACHER_CKPT or None,
                     **MT_OVERRIDES.get("distill", {}))
CFG = WORK / "train_cfg.yaml"
CFG.write_text(yaml.safe_dump({"train": train, "mt": mt}, sort_keys=False))

last = resumable() if RESUME else None
t0 = time.time()
if last:
    print(f"resuming {last.parent.parent.name} from {last}")
    sh(f"{q(sys.executable)} -m adas_mt train --data {q(str(RUN_DATA))} --resume {q(str(last))} --device {q(DEVICE)}")
else:
    weights = attempt_download_asset(str(WORK / f"yolo26{SCALE}.pt"))  # COCO-pretrained trunk (heads start fresh)
    sh(f"{q(sys.executable)} -m adas_mt train --data {q(str(RUN_DATA))} --model {q(str(weights))} --cfg {q(str(CFG))} "
       f"--scale {SCALE} --device {q(DEVICE)} --project {q(str(PROJECT))} --name {q(RUN_NAME)}")
RUN_DIR = run_dirs()[-1]
BEST = RUN_DIR / "weights" / "best.pt"
print(f"\ntraining finished in {(time.time() - t0) / 60:.0f} min -> {RUN_DIR}")

## 5 · Early results
Validation metrics per epoch from `results.csv`, the best epoch (the one `best.pt` comes from, by the run's fitness:
0.5 mAP50-95 + 0.25 drivable-area mIoU + 0.25 lane IoU), and the validation images with predictions.
Lane IoU is measured on the ~4 px-wide training masks; compare runs of this repository with each other, not with
papers that score 2 px lanes.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator

res = pd.read_csv(RUN_DIR / "results.csv")
res.columns = [c.strip() for c in res.columns]
w = yaml.safe_load((RUN_DIR / "mt.yaml").read_text())["fitness"]
res["fitness"] = w["det"] * res["metrics/mAP50-95(B)"] + w["da"] * res["metrics/da_mIoU"] + w["ll"] * res["metrics/ll_IoU_fg"]
best = res.loc[res["fitness"].idxmax()]
show = {"mAP50": "metrics/mAP50(B)", "mAP50-95": "metrics/mAP50-95(B)", "precision": "metrics/precision(B)",
        "recall": "metrics/recall(B)", "DA mIoU": "metrics/da_mIoU", "DA IoU direct": "metrics/da_IoU_direct",
        "DA IoU alternative": "metrics/da_IoU_alternative", "DA IoU (any drivable)": "metrics/da_IoU_fg",
        "lane IoU (any lane)": "metrics/ll_IoU_fg", "lane recall": "metrics/ll_recall_fg",
        "lane IoU solid": "metrics/ll_IoU_solid", "lane IoU dashed": "metrics/ll_IoU_dashed"}
table = pd.DataFrame({"first epoch": [res[c].iloc[0] for c in show.values() if c in res],
                      f"best (epoch {int(best['epoch'])})": [best[c] for c in show.values() if c in res],
                      "last epoch": [res[c].iloc[-1] for c in show.values() if c in res]},
                     index=[k for k, c in show.items() if c in res])
print(table.round(3).to_string())

# small multiples, one measure family per panel (no shared or dual axes); palette slots 1-2, validated for CVD
SURFACE, INK, MUTED, GRID = "#fcfcfb", "#0b0b0b", "#52514e", "#e6e5e1"
S1, S2 = "#2a78d6", "#eb6834"
panels = [("Detection", [("mAP50", "metrics/mAP50(B)"), ("mAP50-95", "metrics/mAP50-95(B)")]),
          ("Drivable area", [("mIoU", "metrics/da_mIoU"), ("IoU any drivable", "metrics/da_IoU_fg")]),
          ("Lanes", [("IoU any lane", "metrics/ll_IoU_fg"), ("recall", "metrics/ll_recall_fg")]),
          ("Train loss: detection", [("box", "train/box_loss"), ("cls", "train/cls_loss")]),
          ("Train loss: segmentation", [("drivable area", "train/da_loss"), ("lane", "train/ll_loss")]),
          ("Distillation (cosine similarity)" if "train/kd_cos" in res else "Val loss: segmentation",
           [("student-teacher", "train/kd_cos")] if "train/kd_cos" in res else [("drivable area", "val/da_loss"), ("lane", "val/ll_loss")])]
fig, axes = plt.subplots(2, 3, figsize=(15, 7.5), facecolor=SURFACE)
for ax, (title, series) in zip(axes.flat, panels):
    ax.set_facecolor(SURFACE)
    present = [(label, col, color) for (label, col), color in zip(series, (S1, S2)) if col in res]
    for label, col, color in present:
        ax.plot(res["epoch"], res[col], color=color, lw=2, label=label)
    ends = [res[col].iloc[-1] for _, col, _ in present]
    lo, hi = ax.get_ylim()
    if len(ends) < 2 or abs(ends[0] - ends[1]) > 0.08 * (hi - lo):  # direct-label the ends only where they do not collide
        for (label, col, _), y in zip(present, ends):
            ax.annotate(f"{label} {y:.3f}", (res["epoch"].iloc[-1], y), xytext=(4, 0), textcoords="offset points",
                        va="center", fontsize=9, color=INK)
    ax.axvline(best["epoch"], color=MUTED, lw=1, ls=(0, (3, 3)))
    ax.set_title(title, loc="left", fontsize=11, color=INK)
    ax.set_xlabel("epoch", color=MUTED, fontsize=9)
    ax.grid(color=GRID, lw=0.8)
    ax.tick_params(colors=MUTED, labelsize=8)
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))
    for s in ax.spines.values():
        s.set_visible(False)
    if len(series) > 1:
        ax.legend(frameon=False, fontsize=8, labelcolor=INK, loc="best")
    first, last_ep = res["epoch"].min(), res["epoch"].max()
    ax.set_xlim(first - 0.5, last_ep + max(1.0, 0.25 * (last_ep - first)))  # room on the right for the end labels
fig.suptitle(f"{RUN_DIR.name}: validation metrics and losses per epoch (dashed: best epoch {int(best['epoch'])})",
             x=0.01, ha="left", color=INK, fontsize=12)
fig.tight_layout()
plt.show()

In [ ]:
from IPython.display import Image as IPImage, display

for name in ("val_batch0_labels.jpg", "val_batch0_pred.jpg"):
    f = RUN_DIR / name
    if f.exists():
        print(name.replace(".jpg", "").replace("val_batch0_", "validation batch: ").replace("labels", "ground truth")
              .replace("pred", "predictions"))
        display(IPImage(filename=str(f), width=1100))

## 6 · Export and check the deployed graph
Exports `best.pt` to ONNX (fused, NMS-free top-k and the mask argmax inside the graph: what TensorRT will run on the
Jetson), then scores the `.pt` and the `.onnx` on the same val images. The two columns should match; a gap means the
export changed the model. Finally, full-resolution overlays from the exported model.

In [ ]:
from adas_mt.deploy.evaluate import evaluate
from adas_mt.engine.val import run_validation
from adas_mt.export import export_onnx

probe_image = val_ims[0]
exp = export_onnx(BEST, verify_image=str(probe_image))
print(f"ONNX: {exp.onnx} ({exp.onnx.stat().st_size / 2**20:.1f} MB) | parity on {probe_image.name}: "
      + ", ".join(f"{k} {v:.3g}" for k, v in exp.parity.items()))
pt_stats, _ = run_validation(BEST, EVAL_DATA, batch=8, device=DEVICE)
onnx_stats, speed = evaluate(exp.onnx, EVAL_DATA, batch=1, device="cpu", backend="ort", workers=2)
keys = ["metrics/mAP50(B)", "metrics/mAP50-95(B)", "metrics/da_mIoU", "metrics/ll_IoU_fg", "metrics/ll_recall_fg"]
cmp = pd.DataFrame({"PyTorch best.pt": [pt_stats[k] for k in keys], "ONNX (ORT, CPU)": [onnx_stats[k] for k in keys]},
                   index=[k.replace("metrics/", "").replace("(B)", "") for k in keys])
cmp["difference"] = cmp.iloc[:, 1] - cmp.iloc[:, 0]
print(f"\nsame {int(speed['images'])} val images:")
print(cmp.round(4).to_string())

In [ ]:
from adas_mt.deploy.runner import predict

demo = WORK / "demo_frames"
shutil.rmtree(demo, ignore_errors=True)
demo.mkdir()
for p in random.Random(1).sample(val_ims, min(6, len(val_ims))):
    shutil.copy(p, demo / p.name)
out = RUN_DIR / "predictions"
stats = predict(exp.onnx, demo, out, conf=0.25, backend="ort", device="cpu")
frames = sorted(out.glob("*.jpg"))
fig, axes = plt.subplots((len(frames) + 1) // 2, 2, figsize=(15, 4.3 * ((len(frames) + 1) // 2)), facecolor="#fcfcfb")
for ax in axes.flat:
    ax.axis("off")
for ax, f in zip(axes.flat, frames):
    ax.imshow(plt.imread(f))
    ax.set_title(f.name, fontsize=9, loc="left")
fig.suptitle("Exported model on val frames: boxes, drivable area (direct / alternative), lanes (solid / dashed)",
             x=0.01, ha="left", fontsize=12)
fig.tight_layout()
plt.show()

## 7 · Where everything is, and next steps
* **Resume** after a disconnect: same `RUN_NAME`, `RESUME = True`, *Run all* (needs `USE_DRIVE`).
* **Full training**: `PRESET = "full"`, a new `RUN_NAME`; on a T4 expect many hours, so keep `USE_DRIVE` on.
  For a large export set `SAVE_CONVERTED_TO_DRIVE` once, then use `DATA_SOURCE = "converted_archive"`.
* **Distillation** (`DISTILL = True`): compare against the same run without it (the A1 vs A2 ablation in `docs/training.md`).
* **Loss balance**: `loss_gains` in `adas_mt/cfg/default.yaml` (`da: 2-5, ll: 2-10`) is the first knob to try if lane
  IoU lags while mAP is fine.
* **On the Jetson**: copy `best.onnx` + `best.json`, then
  `python -m adas_mt trt-build --onnx best.onnx --precision fp16`,
  `python -m adas_mt eval --model best.engine --data data.yaml`, `python -m adas_mt bench --model best.engine --source frame.jpg`
  (see `docs/deployment_jetson.md`).

In [ ]:
print("run directory:", RUN_DIR)
for f in ("weights/best.pt", "weights/best.onnx", "weights/best.json", "results.csv", "results.png", "mt.yaml", "predictions"):
    p = RUN_DIR / f
    print(f"  {'ok ' if p.exists() else '-- '} {p}")
if not USE_DRIVE and IN_COLAB:
    print("\nUSE_DRIVE is off: /content is wiped when the runtime is recycled. Download the files above now:")
    print(f"  from google.colab import files; files.download('{RUN_DIR / 'weights' / 'best.onnx'}')")